In [ ]:
#Model Training 

# Import necessary libraries
import os
import gc
import numpy as np
import pydicom
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Configuration settings
IMG_SIZE = (128, 128)  # Resize images to 128x128 to reduce memory usage
BATCH_SIZE = 16        # Small batch size to fit RTX 4050's 6GB memory
EPOCHS = 20            # Maximum epochs, with early stopping to reduce training time
AUTO = tf.data.AUTOTUNE  # Optimize data pipeline performance

# Function to load and preprocess a single DICOM image
def load_dicom(path):
    """
    Loads a DICOM file, applies windowing, normalizes pixel values, and resizes the image.
    
    Args:
        path: Tensor or string path to the DICOM file
    
    Returns:
        Processed image as a numpy array, or a zero tensor if loading fails
    """
    try:
        # Convert tensor path to string if necessary
        path = path.numpy().decode('utf-8') if isinstance(path, tf.Tensor) else path
        dicom = pydicom.dcmread(path, force=True)
        if not hasattr(dicom, 'pixel_array'):
            return np.zeros((*IMG_SIZE, 1), dtype=np.float32)
        img = dicom.pixel_array.astype(np.float32)
        # Apply brain-specific windowing (center=40, width=80)
        center = dicom.get('WindowCenter', 40)
        width = dicom.get('WindowWidth', 80)
        img = np.clip(img, center - width / 2, center + width / 2)
        # Normalize to [0, 1]
        img = (img - img.min()) / (img.max() - img.min() + 1e-7)
        # Resize to target size
        img = cv2.resize(img, IMG_SIZE)
        return img[..., np.newaxis]  # Add channel dimension
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return np.zeros((*IMG_SIZE, 1), dtype=np.float32)  # Return default tensor on failure

# Function to create a TensorFlow dataset
def create_dataset(paths, labels=None):
    """
    Creates a TensorFlow dataset for efficient data loading.
    
    Args:
        paths: List of file paths to DICOM images
        labels: List of corresponding labels (0 for normal, 1 for hemorrhage), or None for inference
    
    Returns:
        A batched and prefetched tf.data.Dataset
    """
    def _process_path(path):
        img = tf.py_function(load_dicom, [path], tf.float32)
        img.set_shape((*IMG_SIZE, 1))
        return img

    if labels is not None:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(lambda p, l: (_process_path(p), l), num_parallel_calls=AUTO)
        ds = ds.filter(lambda x, l: tf.math.reduce_sum(x) > 0)  # Filter out all-zero images
    else:
        ds = tf.data.Dataset.from_tensor_slices(paths)
        ds = ds.map(_process_path, num_parallel_calls=AUTO)
        ds = ds.filter(lambda x: tf.math.reduce_sum(x) > 0)  # Filter out all-zero images
    
    return ds.batch(BATCH_SIZE).prefetch(AUTO)

# Function to build the model
def build_model():
    """
    Builds a lightweight CNN model using MobileNetV2 for binary classification.
    
    Returns:
        Compiled Keras model
    """
    base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), weights='imagenet', include_top=False)
    base_model.trainable = False  # Freeze base model
    
    model = models.Sequential([
        layers.Conv2D(3, (3, 3), padding='same', input_shape=(*IMG_SIZE, 1)),  # Adapt 1-channel to 3-channel
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Function to apply data augmentation
def augment_dataset(ds):
    """
    Applies data augmentation to the training dataset.
    
    Args:
        ds: TensorFlow dataset
    
    Returns:
        Augmented dataset
    """
    augmentation = models.Sequential([
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1)
    ])
    return ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTO)

# Main training function
def train_model(normal_path, hemorrhage_path):
    """
    Trains the model on normal and hemorrhage DICOM images and saves it.
    
    Args:
        normal_path: Path to normal DICOM directory
        hemorrhage_path: Path to hemorrhage DICOM directory
    
    Returns:
        Trained model and training history
    """
    # Collect file paths and labels
    normal_files = [(os.path.join(r, f), 0) for r, _, fs in os.walk(normal_path) for f in fs if f.endswith('.dcm')]
    hemo_files = [(os.path.join(r, f), 1) for r, _, fs in os.walk(hemorrhage_path) for f in fs if f.endswith('.dcm')]
    all_files = normal_files + hemo_files
    paths, labels = zip(*all_files)
    
    print(f"Normal files found: {len(normal_files)}")
    print(f"Hemorrhage files found: {len(hemo_files)}")
    
    # Split data into training and validation sets
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, labels, test_size=0.1, stratify=labels, random_state=42
    )
    
    # Create datasets
    train_ds = create_dataset(train_paths, train_labels)
    val_ds = create_dataset(val_paths, val_labels)
    train_ds = augment_dataset(train_ds)
    
    # Build and train the model
    model = build_model()
    early_stopping = callbacks.EarlyStopping(patience=5, monitor='val_accuracy', restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    # Save the trained model
    model.save('hemorrhage_model.h5')
    print("Model saved as 'hemorrhage_model.h5'")
    
    # Store validation data for evaluation
    global val_p, val_l
    val_p, val_l = val_paths, val_labels
    
    return model, history

# Evaluation function
def evaluate_model(model):
    """
    Evaluates the model on the validation set.
    
    Args:
        model: Trained Keras model
    """
    val_ds = create_dataset(val_p, val_l)
    preds = model.predict(val_ds)
    preds_binary = (preds > 0.5).astype(int)
    print("\nClassification Report:")
    print(classification_report(val_l, preds_binary, target_names=['Normal', 'Hemorrhage']))

# Plotting function
def plot_history(history):
    """
    Plots training and validation accuracy.
    
    Args:
        history: Training history object
    """
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

# Execute the script
normal_path =  r"C:\Users\Atharva Badgujar\OneDrive\Desktop\Hemorrhage new\LSTM model\Normal"
hemorrhage_path =r"C:\Users\Atharva Badgujar\OneDrive\Desktop\Hemorrhage new\LSTM model\HEMORRHAGES CT AI\Hemorrhage" 

print("Starting model training...")
model, history = train_model(normal_path, hemorrhage_path)
print("Evaluating model performance...")
evaluate_model(model)
print("Plotting training history...")
plot_history(history)

# Clean up memory
tf.keras.backend.clear_session()
gc.collect()
print("Training complete.")

In [ ]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import pydicom
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# Configuration settings
IMG_SIZE = (128, 128)  # Same as used during training
BATCH_SIZE = 16        # Same as used during training

# Hardcoded path to the test dataset folder
TEST_DATASET_PATH = r"C:\Users\Atharva Badgujar\OneDrive\Desktop\Hemorrhage new\LSTM model\TEST"  # Replace with your actual test dataset path

# Function to load and preprocess a single DICOM image for prediction
def load_dicom_for_prediction(path):
    try:
        # Read the DICOM file
        dicom = pydicom.dcmread(path, force=True)
        if not hasattr(dicom, 'pixel_array'):
            print(f"No pixel data in {path}")
            return None
        
        # Get the pixel array and convert to float32
        img = dicom.pixel_array.astype(np.float32)
        
        # If the image has more than 2 dimensions, take the first channel
        if len(img.shape) > 2:
            img = img[..., 0]  # Select the first channel, making it 2D
        
        # Apply windowing (adjust based on your DICOM metadata)
        center = dicom.get('WindowCenter', 40)
        width = dicom.get('WindowWidth', 80)
        img = np.clip(img, center - width / 2, center + width / 2)
        
        # Normalize to [0, 1]
        img = (img - img.min()) / (img.max() - img.min() + 1e-7)
        
        # Resize to target size
        img = cv2.resize(img, IMG_SIZE)
        
        # Add the channel dimension
        img = img[..., np.newaxis]
        
        # Verify the shape
        if img.shape != (128, 128, 1):
            print(f"Unexpected shape after processing {path}: {img.shape}")
            return None
            
        return img
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None
# Prediction function based on confidence level
def predict_with_confidence(model, patient_images, confidence_threshold=0.7):
    """
    Predicts whether a patient has hemorrhage based on image-level predictions
    and a confidence threshold.

    Args:
        model: Trained Keras model.
        patient_images: List of preprocessed images for a patient.
        confidence_threshold: Minimum confidence level to classify an image as positive.

    Returns:
        A tuple containing:
        - Prediction result (1 for hemorrhage, 0 for normal)
        - List of confidence scores for each image
    """
    if not patient_images:
        return 0, []  # Default to normal if no valid images

    # Stack images into a batch for prediction
    batch = np.stack(patient_images, axis=0)
    preds = model.predict(batch, verbose=0).flatten()

    # Calculate average confidence score
    average_confidence = np.mean(preds)

    # Classify as hemorrhage if average confidence exceeds threshold
    prediction = 1 if average_confidence > confidence_threshold else 0

    # Log individual predictions for debugging
    print(f"Image predictions: {preds}")
    print(f"Average confidence: {average_confidence}")
    print(f"Patient-level classification: {'Hemorrhage' if prediction == 1 else 'Normal'}")

    return prediction, preds

# Main prediction function
def predict_and_generate_report(test_dir=TEST_DATASET_PATH, model_path='hemorrhage_model.h5', output_csv='patient_diagnosis_t1.csv', confidence_threshold=0.7):
    """
    Processes the test dataset, predicts hemorrhage vs. normal for each patient using
    confidence-based predictions, and saves the results in a CSV.

    Args:
        test_dir: Path to the test dataset folder containing patient folders.
        model_path: Path to the trained model file.
        output_csv: Path to save the output CSV report.
        confidence_threshold: Minimum confidence level to classify an image as positive.
    """
    # Load the trained model
    model = load_model(model_path)

    # Initialize results list
    results = []

    # Walk through the test directory
    for root, dirs, files in os.walk(test_dir):
        if not files:
            continue  # Skip directories without files

        patient_id = Path(root).name  # Assume patient ID is the folder name
        print(f"\nProcessing patient: {patient_id}")

        # Collect all DICOM files for the patient
        dicom_files = [os.path.join(root, f) for f in files if f.endswith('.dcm')]

        # Preprocess images
        patient_images = [load_dicom_for_prediction(f) for f in dicom_files]
        patient_images = [img for img in patient_images if img is not None]  # Filter out invalid images

        # If no valid images are found, default to 'Normal'
        if not patient_images:
            print(f"No valid images found for patient {patient_id}")
            for dicom_file in dicom_files:
                results.append({'PatientID': patient_id, 'DICOMFile': dicom_file, 'Confidence': None, 'Diagnosis': 'Normal'})
            continue

        # Use predictions for each individual image and store results
        preds = model.predict(np.stack(patient_images), verbose=0).flatten()

        for dicom_file, confidence in zip(dicom_files, preds):
            diagnosis = 'Hemorrhage' if confidence > confidence_threshold else 'Normal'
            results.append({'PatientID': patient_id, 'DICOMFile': dicom_file, 'Confidence': confidence, 'Diagnosis': diagnosis})

    # Save results to CSV
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\nDiagnosis report saved to {output_csv}")


    # Save results to CSV
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\nDiagnosis report saved to {output_csv}")

if _name_ == "_main_":
    predict_and_generate_report()